In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Semantic Chunking

Instead of cutting at a fixed size, `SemanticChunker` **embeds each sentence** and places a break where the embedding similarity between consecutive sentences drops — so each chunk holds a coherent idea and chunk sizes vary with the content.

Trade-off: it calls the embedding model while splitting (extra cost/latency), unlike the character/token splitters. We use `OpenAIEmbeddings` here.

> Applied to the PDF only — semantic chunking embeds every sentence, so running it on the large/noisy HTML page would be slow and costly. The same code works on `html_docs`.

## 1. Load source documents

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
pdf_docs = PyPDFLoader(str(pdf_path)).load()
print(f"PDF: loaded {len(pdf_docs)} page(s)")

/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_62157/2839505774.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF: loaded 12 page(s)


## 2. Semantic chunking

`breakpoint_threshold_type` controls *how aggressively* it breaks:
- `percentile` (default) — break when the similarity gap exceeds the Nth percentile
- `standard_deviation`, `interquartile`, `gradient` — alternative statistics over the gaps

In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
)

# Embeds sentences and splits at semantic boundaries (makes embedding API calls).
chunks = splitter.split_documents(pdf_docs)
print(f"Split {len(pdf_docs)} pages into {len(chunks)} semantic chunks")

/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_62157/3038680049.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Split 12 pages into 23 semantic chunks


Chunk sizes vary (driven by meaning, not a fixed length):

In [4]:
sizes = [len(c.page_content) for c in chunks]
print(f"chunk size (chars) -> min {min(sizes)}, max {max(sizes)}, avg {sum(sizes)//len(sizes)}")
print(f"\n--- first chunk ({sizes[0]} chars) ---\n{chunks[0].page_content[:400]}")

chunk size (chars) -> min 150, max 1968, avg 802

--- first chunk (360 chars) ---
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage. Author: Prabhukumar Sivamoorthy
Prabhukumarsivamoorthy@gmail.com
